# 06 — Entrenamiento exclusivo del mask decoder

Solo Colab GPU, ejecutado por el usuario. Leer docs/project/validacion_etapa6.md. Etapa abierta hasta revisión de evidencia; sin reservados, ablaciones adicionales ni demo.


## Protocolo previo
10 épocas por régimen, semilla 42, AdamW LR 1e-5 / WD 0.01, FP32, batch efectivo 4 por acumulación. SAM base independiente, encoders congelados, DINO no cargado: features/probe propios de etapa 5. Prompts fijos. BCE+Dice válida de unión suave por máximo; sin componentes = vacío sin gradiente. Best por IoU agua val, primer empate. Se conservan resultados negativos.


## 1. Paquete y entorno
Exportar scripts/export_stage6_bundle.py en la PC; subir ZIP a MyDrive y copiar SHA. GPU requerida. Raíz histórica íntegra de Drive. Nunca instalar ni ejecutar modelos en la PC.


In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, zipfile
from google.colab import drive

drive.mount('/content/drive')
PERSIST_ROOT = '/content/drive/MyDrive/auto-prompt-sam2'
REPO = Path('/content/auto-prompt-sam2-stage6')
PYTHON = Path('/content/auto-prompt-venv/bin/python')
BUNDLE = Path('/content/drive/MyDrive/auto-prompt-sam2-stage6.zip')
EXPECTED_BUNDLE_SHA256 = ''  # Copiar salida de export_stage6_bundle.py.
assert len(EXPECTED_BUNDLE_SHA256) == 64, 'Configura el SHA-256 del paquete.'
assert hashlib.sha256(BUNDLE.read_bytes()).hexdigest() == EXPECTED_BUNDLE_SHA256
REPO.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(BUNDLE) as archive:
    for item in archive.infolist():
        assert (REPO / item.filename).resolve().is_relative_to(REPO.resolve())
    archive.extractall(REPO)
child_env = dict(os.environ, PYTHONPATH=str(REPO / 'src'),
    AUTO_PROMPT_PERSIST_ROOT=PERSIST_ROOT, PYTHONHASHSEED='42',
    CUBLAS_WORKSPACE_CONFIG=':4096:8',
    AUTO_PROMPT_EXECUTION_CONTEXT='google_colab_user_notebook')
# Token opcional para descarga autorizada, nunca imprimirlo ni incluirlo en informes.
from google.colab import userdata
try:
    child_env['HF_TOKEN'] = userdata.get('HF_TOKEN')
except userdata.SecretNotFoundError:
    print('HF_TOKEN no configurado; se usara la cache autorizada si existe.')
LOG_DIR = Path(PERSIST_ROOT) / 'stage6/notebook_logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)
def command(args, log_name):
    import time
    log_path = LOG_DIR / (str(time.time_ns()) + '_' + log_name)
    print('Log persistente:', log_path)
    with log_path.open('x', encoding='utf-8') as log:
        process = subprocess.Popen(args, cwd=REPO, env=child_env, text=True,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
        for line in process.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
        result = process.wait()
    if result:
        raise RuntimeError(f'Proceso terminado con código {result}; conservar salida y log {log_path}')
if not PYTHON.is_file():
    command([sys.executable, str(REPO / 'scripts/bootstrap_colab.py')], 'bootstrap.log')
command([str(PYTHON), '-m', 'pip', 'install', '--no-deps', '-r', str(REPO / 'requirements-stage2.txt')], 'stage2_pins.log')
command([str(PYTHON), '-m', 'pip', 'check'], 'pip_check.log')


## 2. Interrupción controlada
Ejecutar prepare UNA VEZ. Una época por régimen, con last persistente. Guardar notebook con salidas; después desconectar y eliminar runtime. No ejecutar resume en este runtime.


In [ ]:
command([str(PYTHON), str(REPO / 'scripts/run_stage6.py'), 'prepare', '--root', PERSIST_ROOT], 'prepare.log')


## 3. Nuevo runtime: reanudar
En otro runtime GPU, ejecutar SOLO la celda de entorno y esta celda. Conservar salidas de prepare. Misma raíz, SHA y código; anterior escritor detenido. Verifica estados y repetición del siguiente paso, revierte diagnóstico y completa presupuesto. Ante interrupción posterior repetir resume; se repite época no comprometida.


In [ ]:
command([str(PYTHON), str(REPO / 'scripts/run_stage6.py'), 'resume', '--root', PERSIST_ROOT], 'resume.log')


## 4. Exportar evidencia y detener
Compartir notebook con salidas, ambos evidence.zip y SHA, logs/failure si existen. Checkpoints quedan en Drive con inventario; facilitar referencias para revisión. No cerrar etapa sin revisión externa.


In [ ]:
for regime in ('official', 'leave_paraguay_out'):
    for folder in sorted((Path(PERSIST_ROOT) / 'regimes' / regime / 'stage6').iterdir()):
        report = folder / 'evidence/validation.json'
        if report.is_file():
            value = json.loads(report.read_text())
            print(regime, value['status'], 'best:', value['progress']['best_epoch'], value['progress']['best_iou'])
            print('Recovery:', value['progress']['recovery'])
            archive = folder / 'evidence.zip'
            print(archive, hashlib.sha256(archive.read_bytes()).hexdigest())
